# 6주차 주관식 문제 - 풀이 (해설 포함)

각 문제의 의도·핵심 개념·함정을 정리.

## 1. 데이터 추출 에이전트의 디코딩 최적화

### 문제 의도
JSON 파싱 에러를 없애려면 모델 출력의 **무작위성**을 제거해야 한다. OpenAI Chat API 기준 관련 파라미터는 `temperature`, `top_p`, `presence_penalty`, `frequency_penalty` 4가지.

### 핵심 개념
| 파라미터 | 기능 | 결정론적 값 |
|---|---|---|
| `temperature` | logits 스케일. 낮을수록 peak 집중. 0 → Greedy. | **0.0** |
| `top_p` | Nucleus Sampling. 누적확률 p 내에서 추출. | **1.0** (disable) |
| `presence_penalty` | 이미 나온 토큰 재등장 억제. | **0.0** (일관성 유지) |

> ⚠️ OpenAI API에서 `top_p=0.0`은 사실상 정의되지 않은 구역이다. 결정론적 의도를 깔끔하게 표현하려면 **`top_p=1.0` + `temperature=0.0`** 조합이 안전하다. temperature=0이면 top_p 값과 무관하게 argmax가 선택된다.

### 정답 코드
```python
{"temperature": 0.0, "top_p": 1.0, "presence_penalty": 0.0}
```


## 2. Few-shot을 활용한 구조적 데이터 추출 최적화

### 문제 의도
Zero-shot으로 LLM에 구조 추출을 시키면 **포맷 오염**(설명 문장 삽입, 스키마 붕괴)이 잦다. Few-shot 예시로 "입·출력 쌍"을 시연해 모델이 패턴 자체를 학습하게 한다.

### 설계 포인트
1. **시스템 프롬프트**: 역할·스키마·금지사항을 선언 ("JSON 배열만 반환, 설명 금지").
2. **Few-shot 예시**: `user`/`assistant` role을 번갈아 2~3개. 다양성 확보(1인·다인·인물 없음 케이스).
3. **메시지 순서**: system → few-shot 쌍들 → 실제 user 질문. 마지막이 테스트 입력이어야 모델이 "이 스타일로 답하면 된다"고 이해.

### 함정
- 예시의 출력을 문자열이 아닌 Python list로 넣으면 API가 이를 그대로 전송하지 못한다. JSON **문자열**로 넣어야 함.
- Few-shot이 너무 유사하면 오버핏팅 → 다양성 필요.


## 3. 구조화 데이터 추출 + Self-Correction

### 문제 의도
파서 수준의 신뢰성을 얻으려면 **(a) 엄격한 포맷 지시** + **(b) 에러 기반 재시도 루프**가 함께 있어야 한다. 이 문제는 시스템 로직(`safe_extract_receipt`)이 에러 메시지를 다시 프롬프트로 주입하는 Self-Correction 구조를 이미 갖추고 있다.

### 시스템 프롬프트 설계
- 역할(Persona): "영수증 파서"
- 스키마 강제: 필수 필드/타입/포맷(ISO date, int amount)
- 포맷 규칙: "JSON만, 코드블록 금지, 쉼표 제거 후 정수"

### User 프롬프트 분기
- 첫 시도: 영수증 텍스트만 전달
- 재시도: `이전 시도 에러 메시지`를 함께 전달 → 모델이 원인을 자각하고 수정

### 기대 결과
```json
{
  "store_name": "맛있는 파스타",
  "date": "2024-03-15",
  "items": [
    {"name": "알리오올리오", "price": 15000},
    {"name": "고르곤졸라 피자", "price": 22000},
    {"name": "콜라 2잔", "price": 6000}
  ],
  "total_amount": 43000
}
```


## 4. Sentiment JSON 프롬프트 e2e

### 문제 의도
자유 텍스트 → 스키마 고정 JSON 변환을 단일 system 프롬프트로 해결. Problem 2,3과 달리 few-shot 없이 **시스템 규칙만으로** 포맷 안정성 확보가 목표.

### 프롬프트 설계 키포인트
1. **역할 선언**: "감성 분석기"
2. **스키마 제시**: `{sentiment, reason}`
3. **enum 강제**: sentiment는 3개 값만 허용
4. **Tiebreaker 규칙**: 감정이 애매할 때 neutral로 분류 → 모델의 자의적 편향 억제
5. **reason 길이 제한**: 한 문장

### 검증
3개 테스트 입력에 대해 각각 positive/neutral/negative로 판정되는지 확인.


## 5. Standard vs CoT 성능 비교

### 문제 의도
다단계 산술 퍼즐(정답 112)에서 **Zero-shot Standard**와 **Chain-of-Thought**의 차이를 실측으로 체감.

### 단계별 정답
| 단계 | 계산 | 재고 |
|---|---|---|
| 월요일 | 100 − 20 + 5 | **85** |
| 화요일 | 85 − floor(85×0.1)=85−8 | **77** |
| 수요일 | 77 + 50 − 15 | **112** |

### CoT가 효과적인 이유
1. 단계별 라벨이 모델의 **short-term working memory**를 프롬프트에 외재화.
2. `floor(x)=y` 같은 연산 포맷이 부동소수점/반올림 혼동을 차단.
3. 최종 답 라벨이 중간 노이즈와 섞이지 않도록 분리.

### 흔한 실수
- 반품(+5)을 추가 출고(-5)로 처리 → 75
- 화요일 10%를 반올림(9) → 76
- 폐기 15를 누락 → 127


## 6. ROUGE / BLEU 핵심 포인트

### 빈칸
torchmetrics `ROUGEScore` 반환 dict의 키는 `rougeL_fmeasure` (및 `rougeL_precision`, `rougeL_recall`).

### 지표 특성
- **ROUGE-L**: 최장 공통 부분수열(LCS) 기반. **Recall** 관점(참조를 얼마나 담았나).
- **BLEU**: n-gram Precision 기반 + brevity penalty.

### 질문 4 — 동의어 치환 시
표면 문자열 매칭 기반이므로 '돕는다↔지원한다' 치환에 **페널티**. 점수 **하락**. BLEU는 n=1~4 동시 일치를 요구해 특히 민감.

### 질문 5 — 극복 방안
- **BERTScore**(임베딩 코사인) — 동의어 수용.
- **BLEURT / COMET** — 사람 평점과 직접 상관.
- **G-Eval / LLM-as-Judge** — 다차원 평가.


## 7. ROUGE-L vs BERTScore

### 목표 검증
```
assert rouge_val < 0.2   # 단어가 거의 안 겹치므로 낮게
assert bert_val  > 0.8   # 의미는 비슷하므로 높게
assert bert_val > rouge_val
```

### 구현 포인트
- `rouge_scorer.RougeScorer(['rougeL']).score(ref, pred)['rougeL'].fmeasure`
- `bert_score.score(cands=[pred], refs=[ref], lang='ko')` — 한국어는 반드시 `lang="ko"` 또는 multilingual 모델 지정.
- 반환은 `(P, R, F1)` 튜플 → `.item()`으로 스칼라화.

### 해석
두 문장의 어휘는 거의 완전히 다르지만(AI/인공지능, 비서/에이전트, 유저/사용자, 작업/업무, 생산성/효율, 향상/높여) 의미는 동일. BERTScore는 이를 반영해 ~0.85 내외, ROUGE-L은 ~0.1 내외가 일반적.


## 8. Contrastive Decoding

### 개념
**Expert**(큰 모델) 분포에서 **Amateur**(작은 모델) 분포를 빼, 두 모델이 공유하는 '뻔한·반복적' 패턴을 억제하고 Expert 고유의 지식을 강조.

$$ \text{CD}(x_t) = \log p_{exp}(x_t) - \tau \cdot \log p_{amt}(x_t) \;\; \text{if } x_t \in V_{head}, \text{else } -\infty $$

### 단계별 구현
1. **Adaptive Masking**: Expert logits를 정규화하여 α(신뢰 하한) 미만은 노이즈로 간주 → `-inf`.
2. **Contrastive Logits**: `expert - τ · amateur`.
3. **Argmax**: 마스크 이후 최댓값의 인덱스.

### 테스트 수학 검증 (α=0.1, τ=1.2)
- Expert logits `[12, 15, 4]` → 정규화 `[0.727, 1.0, 0.0]` → 인덱스 2 마스킹.
- Contrastive = `[12−12, 15−17.4, 4−2.4]` = `[0, −2.4, 1.6]`
- 마스킹 후: `[0, −2.4, −inf]` → **argmax = 0** ✅

### 왜 0이 선택되나
- 인덱스 1은 Expert가 강하게 좋아하지만, Amateur도 거의 같은 확신(14.5)을 보임 → "뻔한 답" → 감산 후 점수 폭락.
- 인덱스 0은 Expert 점수는 중간이지만 Amateur도 같이 높아 가산 효과가 상쇄 → 0으로 유지.
- 인덱스 2는 Expert가 아예 버리는 토큰 → 마스킹.
결과적으로 Amateur와 **덜 겹치는** 인덱스 0이 선택됨.


## 9. DSPy

### 9-1. Signature
- `dspy.InputField` / `dspy.OutputField`에 **의미 있는 `desc`**를 붙여야 DSPy 컴파일러가 지시문을 올바르게 생성한다.
- 출력은 열거형(`Safe`/`Unsafe`)으로 좁혀 파싱 안정성 확보.

### 9-2. ChainOfThought Module
- `dspy.ChainOfThought(Signature)`는 Signature에 **`rationale` 출력을 자동 추가**해, 결과와 함께 추론 과정도 반환한다.
- `forward`에서는 `self.analyze_safety(user_input=...)`만 호출하면 DSPy가 프롬프트 렌더링을 담당.

### 9-3. Decoding Config
- **`top_p=0.95`** — Nucleus Sampling. 누적확률 95% 내 토큰만 후보. 꼬리 확률의 말도 안 되는 토큰(환각 소스) 차단.
- **`repetition_penalty=1.2`** — OSS 모델(Hugging Face) 표준. 이미 등장한 토큰 로짓에 나눗셈 페널티(>1이면 감소) → 반복 루프 억제.

> OpenAI API를 쓸 경우 `repetition_penalty` 대신 `frequency_penalty`/`presence_penalty`(-2~2)를 사용. 강의가 OSS 관점이면 `repetition_penalty`로 충분.


## 10. Self-Correction 루프

### 개념
"한 방 프롬프트"가 못 푸는 다제약 생성 문제를 **(생성 → 비판 → 프롬프트 개정 → 재생성)** 루프로 수렴시키는 메타 프롬프팅.

### Critic Prompt 설계
- 편집장 Persona → 팩트 기반 평가.
- 조건별로 "현재 값 / 판정" **포맷 강제** — 후속 단계에서 기계적으로 재사용 가능한 피드백 구조.
- 마지막에 `종합: PASS/FAIL` 라벨을 붙여 **조기 종료 트리거**로 활용.

### Improvement Prompt 설계
- 편집장 피드백을 **강화된 제약 문장**으로 변환.
- "작성 후 글자 수를 스스로 세어보고 벗어나면 재작성" 같은 **Self-Check 지시** 포함 — 단일 생성 시에도 모델이 조건을 한 번 더 검증하게 유도.
- 원문 지시는 유지하되 위반 조건의 '엄격도'만 상승.

### 최적화 팁
- `종합: PASS`면 루프 조기 종료 → 비용·지연 절약.
- `temperature=0.2`로 재현성 확보하되 필요 시 재시도 간 살짝 변동.
- 최대 루프 횟수(10)는 수렴 안정화 지표로. 실제 서비스에서는 3~5회가 실용적.

### 채점 조건 요약
- 글자 수 120~150
- 필수: 친환경, 노이즈캔슬링, 할인
- 금지: 플라스틱, 최고, 애플
- 맨 끝 해시태그 정확히 3개
